# LOGOS prototype 4: failure warning on real run-to-failure data (NASA C-MAPSS) with a human-readable report

Question: on data we did not simulate ourselves, can a guardian warn before failure with a controlled false-alarm rate, explain why in plain language, and leave the decision to a human?

- Data: NASA C-MAPSS turbofan engine degradation (FD001 simple conditions, FD002 six operating conditions). These are NASA simulations of jet engines, not cars.
- Runtime: Colab, CPU is enough. Steps 2 and 3 take a few minutes.
- Honest limits are at the end of the notebook.

## Step 1: download and look at the data

In [ ]:
import urllib.request, zipfile, os, glob
import numpy as np, pandas as pd

url = "https://kr0k0tsch.de/rul-datasets/CMAPSSData.zip"
if not os.path.exists("cmapss"):
    urllib.request.urlretrieve(url, "cmapss.zip")
    with zipfile.ZipFile("cmapss.zip") as z:
        z.extractall("cmapss")
path = glob.glob("cmapss/**/train_FD001.txt", recursive=True)[0]
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
df = pd.read_csv(path, sep=r"\s+", header=None, names=cols)
life = df.groupby("unit")["cycle"].max()
print("rows:", len(df), "| engines:", df["unit"].nunique())
print("cycles per engine: min %d, median %d, max %d" % (life.min(), life.median(), life.max()))

## Step 2: fair comparison on real engines

Four methods are each set to the same number of false alarms per 1000 healthy cycles. Engines are split by engine (5-fold), so no engine is both training and test. An engine counts as warned if an alarm fires in its last 60 cycles.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

H, W, L = 30, 20, 60

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]
        m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)])
        idx.append(t - 1)
    return np.array(feats), np.array(idx)

def run_file(name, budgets=(1, 3), repeats=2):
    path = glob.glob("cmapss/**/" + name, recursive=True)[0]
    cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
    data = pd.read_csv(path, sep=r"\s+", header=None, names=cols)
    sensors = [c for c in cols[5:] if data[c].nunique() >= 8]      # drop sensors that never change
    d = len(sensors)
    F, y, ep, tpos, fail_times = [], [], [], [], {}
    for u, g in data.groupby("unit"):
        f, idx = window_features(g[sensors].values)
        ft = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tpos.append(idx)
        y.append((((ft - idx) <= H) & ((ft - idx) > 0)).astype(int)); fail_times[u] = ft
    F, y, ep, tpos = np.vstack(F), np.concatenate(y), np.concatenate(ep), np.concatenate(tpos)
    tf = np.array([fail_times[u] for u in ep], dtype=float)
    P = dict(ep=ep, t=tpos, tf=tf, healthy=~(tpos >= tf - L), win=(tpos < tf) & (tpos >= tf - L),
             n_fail=len(fail_times), per1000=(~(tpos >= tf - L)).sum() / 1000.0)

    def evaluate_alarm(score, thr, gap=10):
        alarm = score > thr
        m = alarm & P["healthy"]
        e_, t_ = P["ep"][m], P["t"][m]
        events = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > gap)).sum()) if len(t_) else 0
        mw = alarm & P["win"]
        leads = []
        if mw.any():
            for e in np.unique(P["ep"][mw]):
                k = mw & (P["ep"] == e)
                leads.append(P["tf"][k][0] - P["t"][k].min())
        return len(leads) / P["n_fail"], (np.median(leads) if leads else 0.0), events / P["per1000"]

    def threshold_for_budget(score, target):
        lo, hi = float(score.min()), float(score.max())
        for _ in range(30):
            mid = (lo + hi) / 2
            if evaluate_alarm(score, mid)[2] > target: lo = mid
            else: hi = mid
        return hi

    names = ["best single sensor, current reading", "model on current readings only",
             "model on 20-cycle trends (logistic)", "model on 20-cycle trends (boosting)"]
    allres = []
    for seed in range(repeats):
        rng = np.random.default_rng(seed)
        units = np.array(sorted(fail_times)); rng.shuffle(units)
        S = {n: np.zeros(len(y)) for n in names}
        for fu in np.array_split(units, 5):
            te = np.isin(ep, fu); tr = ~te
            ltr, lte = F[tr][:, d:2*d], F[te][:, d:2*d]
            aucs = np.array([roc_auc_score(y[tr], ltr[:, j]) for j in range(d)])
            j = int(np.argmax(np.abs(aucs - 0.5))); sg = 1 if aucs[j] > 0.5 else -1
            S[names[0]][te] = sg * lte[:, j]
            s1 = StandardScaler().fit(ltr)
            S[names[1]][te] = LogisticRegression(max_iter=3000).fit(s1.transform(ltr), y[tr]).predict_proba(s1.transform(lte))[:, 1]
            s2 = StandardScaler().fit(F[tr])
            S[names[2]][te] = LogisticRegression(max_iter=3000).fit(s2.transform(F[tr]), y[tr]).predict_proba(s2.transform(F[te]))[:, 1]
            S[names[3]][te] = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, random_state=0).fit(F[tr], y[tr]).predict_proba(F[te])[:, 1]
        allres.append({(b, n): evaluate_alarm(S[n], threshold_for_budget(S[n], b)) for b in budgets for n in names})
    print("\n##### %s: %d engines, %d sensors used, %d repeats of 5-fold CV by engine #####" % (name, P["n_fail"], d, repeats))
    for b in budgets:
        print("--- every method set to %g false alarms per 1000 healthy cycles ---" % b)
        for n in names:
            a = np.array([r[(b, n)] for r in allres])
            print("%-38s warned %3.0f%% | median lead %3.0f cycles | false alarms %.1f/1000"
                  % (n, 100 * a[:, 0].mean(), a[:, 1].mean(), a[:, 2].mean()))

run_file("train_FD001.txt")
run_file("train_FD002.txt")

Reading (our run): with a strict false-alarm budget, a single sensor on FD002 warned only 39% of engines while the multi-sensor models warned 100%. A model on current readings of all sensors did about as well as the trend models, so the gain came from combining sensors, not from trends. Differences of a few cycles in lead time are within noise (only 2 repeats).

## Step 3: a warning report a human can read, and a human decision

The model is trained on 80 engines and tested on 20 unseen ones. For an alert it shows which sensors pushed the score up, how far they are from early-life values, a suggested action, and its limits. Nothing happens in the real world: the answer you type is only recorded.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

H, W, L = 30, 20, 60
cols = ["unit", "cycle"] + ["set%d" % i for i in range(1, 4)] + ["s%d" % i for i in range(1, 22)]
data = pd.read_csv(glob.glob("cmapss/**/train_FD001.txt", recursive=True)[0], sep=r"\s+", header=None, names=cols)
sensors = [c for c in cols[5:] if data[c].nunique() >= 8]
d = len(sensors)
what = {"s2": "temperature at the low-pressure compressor outlet", "s3": "temperature at the high-pressure compressor outlet",
        "s4": "temperature at the low-pressure turbine outlet", "s7": "pressure at the high-pressure compressor outlet",
        "s8": "fan speed", "s9": "core speed", "s11": "static pressure at the high-pressure compressor outlet",
        "s12": "fuel flow ratio", "s13": "corrected fan speed", "s14": "corrected core speed", "s15": "bypass ratio",
        "s17": "bleed enthalpy", "s20": "turbine coolant bleed (high-pressure)", "s21": "turbine coolant bleed (low-pressure)"}

def window_features(X):
    feats, idx = [], []
    tt = np.arange(W) - (W - 1) / 2
    for t in range(W, len(X) + 1):
        w = X[t - W:t]; m = w.mean(0)
        feats.append(np.r_[m, w[-1], (tt[:, None] * (w - m)).sum(0) / (tt ** 2).sum(), w.std(0)]); idx.append(t - 1)
    return np.array(feats), np.array(idx)

units = sorted(data["unit"].unique()); cut = int(0.8 * len(units))
train_u, test_u = units[:cut], units[cut:]
def build(us):
    F, y, ep, tp, ft = [], [], [], [], {}
    for u in us:
        g = data[data["unit"] == u]; f, idx = window_features(g[sensors].values); t_f = len(g) - 1
        F.append(f); ep.append(np.full(len(idx), u)); tp.append(idx); ft[u] = t_f
        y.append((((t_f - idx) <= H) & ((t_f - idx) > 0)).astype(int))
    return np.vstack(F), np.concatenate(y), np.concatenate(ep), np.concatenate(tp), ft
Ftr, ytr, etr, ttr, ftr = build(train_u)
Fte, yte, ete, tte, fte = build(test_u)

sc = StandardScaler().fit(Ftr)
lr = LogisticRegression(max_iter=3000).fit(sc.transform(Ftr), ytr)
p_tr, p_te = lr.predict_proba(sc.transform(Ftr))[:, 1], lr.predict_proba(sc.transform(Fte))[:, 1]

# alarm threshold chosen on TRAINING engines: about 1 false alarm per 1000 healthy cycles
tf_tr = np.array([ftr[u] for u in etr]); healthy = ttr < tf_tr - L
def fa_rate(thr):
    m = (p_tr > thr) & healthy; e_, t_ = etr[m], ttr[m]
    ev = 1 + int(((np.diff(e_) != 0) | (np.diff(t_) > 10)).sum()) if len(t_) else 0
    return ev / (healthy.sum() / 1000)
lo, hi = 0.0, 1.0
for _ in range(30):
    mid = (lo + hi) / 2
    if fa_rate(mid) > 1: lo = mid
    else: hi = mid
thr = hi

# "normal" reference: early life (first 40 cycles) of training engines
early = data[(data["unit"].isin(train_u)) & (data["cycle"] <= 40)][sensors]
mu, sd = early.mean().values, early.std().values + 1e-9

def report(u):
    m = ete == u; p, t, X, Xs = p_te[m], tte[m], Fte[m], sc.transform(Fte[m])
    hit = np.where(p > thr)[0]
    if len(hit) == 0:
        return print("Engine %d: no alarm was raised." % u)
    k = hit[0]
    contrib = (lr.coef_[0] * Xs[k]).reshape(4, d).sum(0)          # how much each sensor pushed the risk up
    z = (X[k][:d] - mu) / sd                                       # recent average vs normal, in standard deviations
    print("=" * 64)
    print("LOGOS ALERT  |  engine %d  |  cycle %d" % (u, t[k] + 1))
    print("Warning score: %.0f%%  (alarm level: %.0f%%). It is a score for 'failure within %d cycles', not an exact probability." % (100 * p[k], 100 * thr, H))
    print("Why (the three sensors that pushed the estimate up the most):")
    for j in np.argsort(-contrib)[:3]:
        print("  - %s (%s): recent average is %.1f standard deviations %s than in early life"
              % (sensors[j], what.get(sensors[j], "sensor"), abs(z[j]), "higher" if z[j] > 0 else "lower"))
    print("Suggested action: schedule an inspection soon. Do not wait for the next routine check.")
    print("Limits: this is a statistical pattern seen before failures in other engines, not a diagnosis.")
    print("        Some failures give no warning at all. A human decides.")
    print("(For checking only, hidden from the system: the engine actually failed at cycle %d, %d cycles after this alert.)"
          % (fte[u] + 1, fte[u] - t[k]))
    try:
        a = input("Approve the inspection? type y or n, then Enter: ").strip().lower()
    except EOFError:
        a = "n"
    print("Decision recorded:", "APPROVED by human" if a == "y" else "NOT approved (no action taken)")

for u in test_u[:2]:
    report(u)

## Limits (please read)

- NASA C-MAPSS is a simulation of jet engines, not cars and not real fleet logs.
- Failure prediction from sensors (remaining useful life) is an established field. The LOGOS project focus is the system around it: knowing the limits of what the system knows, explaining in plain language, and human approval.
- The alarm threshold in Step 3 is set on training engines; the 'warning score' is not a calibrated probability.
- An alert more than 60 cycles before failure is counted as a false alarm in Step 2 although it may be a useful early warning. The metric is strict on purpose.
- No safety assessment of any kind.